In [ ]:
from typing import TypedDict
from rich import print

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph,START,END
from loguru import logger

#1. 状態を宣言
class OverAllState(TypedDict):
    final_res:str

#2. ノードを宣言
def node_a(state:OverAllState)->OverAllState:
    logger.info("node_a が実行されました")
    return {
        "final_res":"node_a 実行の中間結果"
    }

def node_b(state:OverAllState)->OverAllState:
    logger.info("node_b が実行されました")
    return {
        "final_res":"node_b 実行の中間結果"
    }

def node_c(state:OverAllState)->OverAllState:
    logger.info("node_c が実行されました")
    return {
        "final_res":"node_c 実行の結果"
    }

#3. グラフを構築
builder = StateGraph(state_schema=OverAllState)
builder.add_node("node_a",node_a)
builder.add_node("node_b",node_b)
builder.add_node("node_c",node_c)

builder.add_edge(START,"node_a")
builder.add_edge("node_a","node_b")
builder.add_edge("node_b","node_c")
builder.add_edge("node_c",END)

#4. チェックポインターバックエンドストレージ
checkpointer = InMemorySaver()
graph = builder.compile(
    checkpointer=checkpointer,
    interrupt_before=["node_a","node_b"],
    interrupt_after=["node_a","node_b"]
)

from IPython.display import display
display(graph)

config = {"configurable":{"thread_id":"123"}}


In [ ]:
logger.info("==========1回目の実行==============")
first_res = graph.invoke({},config=config)
logger.info("1回目の実行結果:{}",first_res)


In [ ]:
logger.info("==========2回目の実行==============")
#中断からの再開実行では状態を None に指定する必要がある
second_res = graph.invoke(None,config=config)
logger.info("2回目の実行結果:{}",second_res)


In [ ]:
logger.info("==========3回目の実行==============")
#中断からの再開実行では状態を None に指定する必要がある
res3 = graph.invoke(None,config=config)
logger.info("3回目の実行結果:{}",res3)


In [ ]:
logger.info("==========4回目の実行==============")
#中断からの再開実行では状態を None に指定する必要がある
res4 = graph.invoke(None,config=config)
logger.info("4回目の実行結果:{}",res4)
